In [2]:
docs_name = [
    'Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt',
    'Datasets/Unstructured_Data/3okobat/3okobat_num36_2020.txt',
    'Datasets/Unstructured_Data/3okobat/3okobat.pdf',
    'Datasets/Unstructured_Data/Asle7a/asle7a.txt',
    'Datasets/Unstructured_Data/child/child.txt',
    'Datasets/Unstructured_Data/dostor_gena2y/dostor.pdf',
    'Datasets/Unstructured_Data/drugs/drugs.txt',
    'Datasets/Unstructured_Data/egra2at_gena2ya/egra2at_gena2ya.pdf',
    'Datasets/Unstructured_Data/erhab/erhab.txt',
    'Datasets/Unstructured_Data/na2d/mbade2_ma7kmt_elna2d.pdf',
    'Datasets/Unstructured_Data/technology/tech.txt'
]

In [3]:
import fitz
import os
from langchain_classic.schema import Document

base_url = "/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject"

def load_pdf_pymupdf(pdf_path):
    """Load PDF content using PyMuPDF"""
    try:
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text()
        doc.close()
        return text
    except Exception as e:
        print(f"Error loading PDF {pdf_path}: {e}")
        return None

def load_text_file(txt_path):
    """Load text file content"""
    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            return f.read()
    except Exception as e:
        print(f"Error loading text file {txt_path}: {e}")
        return None

documents = []

for doc_name in docs_name:
    full_path = os.path.join(base_url, doc_name)
    ext = os.path.splitext(doc_name)[1].lower()

    content = None
    if ext == ".pdf":
        content = load_pdf_pymupdf(full_path)
    elif ext == ".txt":
        content = load_text_file(full_path)
    else:
        print(f"⚠ Skipping unsupported file type: {doc_name}")

    if content:
        documents.append(
            Document(
                page_content=content,
                metadata={"source": doc_name}
            )
        )
        print(f"✓ Successfully loaded: {doc_name}")
    else:
        print(f"✗ Failed to load: {doc_name}")

print(f"\nTotal loaded: {len(documents)}/{len(docs_name)} documents")


✓ Successfully loaded: Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt
✓ Successfully loaded: Datasets/Unstructured_Data/3okobat/3okobat_num36_2020.txt
✓ Successfully loaded: Datasets/Unstructured_Data/3okobat/3okobat.pdf
✓ Successfully loaded: Datasets/Unstructured_Data/Asle7a/asle7a.txt
✓ Successfully loaded: Datasets/Unstructured_Data/child/child.txt
✓ Successfully loaded: Datasets/Unstructured_Data/dostor_gena2y/dostor.pdf
✓ Successfully loaded: Datasets/Unstructured_Data/drugs/drugs.txt
✓ Successfully loaded: Datasets/Unstructured_Data/egra2at_gena2ya/egra2at_gena2ya.pdf
✓ Successfully loaded: Datasets/Unstructured_Data/erhab/erhab.txt
✓ Successfully loaded: Datasets/Unstructured_Data/na2d/mbade2_ma7kmt_elna2d.pdf
✓ Successfully loaded: Datasets/Unstructured_Data/technology/tech.txt

Total loaded: 11/11 documents


In [4]:
documents

[Document(metadata={'source': 'Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt'}, page_content='المادة الأولى: يستبدل بنص المادة ٣٩٢ من قانون العقوبات الصادر بالقانون رقم ٥٨ لسنة 1937 النص الآتي: مادة ٣٩٢ كل من صدر عليه حكم قضائي واجب النفاذ بدفع نفقة لزوجه أو أقاربه أو أصهاره أو أجرة حضانة أو رضاعة أو مسكن وامتنع عن الدفع مع قدرته عليه لمدة ثلاثة أشهر بعد التنبيه عليه بالدفع، يعاقب بالحبس مدة لا تزيد على سنة وبغرامة لا تتجاوز خمسة آلاف جنيه أو بإحدى هاتين العقوبتين، ولا ترفع الدعوى عليه إلا بناء على شكوى أو طلب من صاحب الشأن، وإذا رفعت بعد الحكم عليه دعوى ثانية عن هذه الجريمة فتكون عقوبته الحبس مدة لا تزيد على سنة، ويترتب على الحكم الصادر بالإدانة تعليق استفادة المحكوم عليه من الخدمات المطلوب الحصول عليها بمناسبة ممارسته نشاطه المهني والتي تقدمها الجهات الحكومية والهيئات العامة ووحدات القطاع العام وقطاع الأعمال العام والجهات التي تؤدي خدمات مرافق عامة، حتى أداء ما تجمد في ذمته لصالح المحكوم له وبنك ناصر الاجتماعي حسب الأحوال.\nالمادة الثانية: ينشر هذا القانون في الجريدة الرسم

In [5]:
import re
from langchain_classic.schema import Document

def legal_split(text, source_name, pattern):
    documents = []

    if 'article' in pattern:
        article_pattern = pattern['article']
        matches = list(re.finditer(article_pattern, text, re.DOTALL | re.MULTILINE))
        
        current_law = ""

        for i, match in enumerate(matches):
            start = match.start()
            end = matches[i+1].start() if i+1 < len(matches) else len(text)
            article_text = text[start:end].strip()

            if len(article_text) < 10:
                continue 

            # رقم المادة
            article_number_match = re.search(r'\d+', match.group())
            article_number = article_number_match.group() if article_number_match else "غير محدد"

            # السياق قبل المادة للبحث عن القانون
            context_text = text[max(0, start-500):start]
            law_match = re.search(pattern.get('law', r''), context_text)
            if law_match:
                current_law = law_match.group()
            
            # استخراج الموجز والقاعدة
            summary_match = re.search(r'الموجز\s*:(.*?)(?=القاعدة|$)', article_text, re.DOTALL)
            summary_text = summary_match.group(1).strip() if summary_match else ""

            rule_match = re.search(r'القاعدة\s*:(.*)', article_text, re.DOTALL)
            rule_text = rule_match.group(1).strip() if rule_match else ""

            # إنشاء الـ Document
            doc = Document(
                page_content=article_text,
                metadata={
                    'source': source_name,
                    'article_number': article_number,
                    'law': current_law,
                    'summary': summary_text,
                    'rule': rule_text
                }
            )
            documents.append(doc)

    else:
        # لو مش مقسم بـ articles، ناخد كل الموضوعات كنصوص كاملة
        topic_pattern = pattern.get('topic', r'.+')
        topic_matches = list(re.finditer(topic_pattern, text, re.MULTILINE))
        for i, match in enumerate(topic_matches):
            start = match.start()
            end = topic_matches[i+1].start() if i+1 < len(topic_matches) else len(text)
            section_text = text[start:end].strip()

            if len(section_text) < 10:
                continue 

            article_number_match = re.search(r'\d+', match.group())
            article_number = article_number_match.group() if article_number_match else "غير محدد"

            context_text = text[max(0, start-500):start]
            law_match = re.search(pattern.get('law', r''), context_text)
            law = law_match.group() if law_match else "غير موجود"

            summary_match = re.search(r'الموجز\s*:(.*?)(?=القاعدة|$)', section_text, re.DOTALL)
            summary_text = summary_match.group(1).strip() if summary_match else ""

            rule_match = re.search(r'القاعدة\s*:(.*)', section_text, re.DOTALL)
            rule_text = rule_match.group(1).strip() if rule_match else ""

            doc = Document(
                page_content=section_text,
                metadata={
                    'source': source_name,
                    'article_number': article_number,
                    'law': law,
                    'summary': summary_text,
                    'rule': rule_text
                }
            )
            documents.append(doc)

    return documents

In [6]:
all_legal_docs = []

normal_split = [
    'Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt',
    'Datasets/Unstructured_Data/3okobat/3okobat_num36_2020.txt',
    'Datasets/Unstructured_Data/3okobat/3okobat.pdf',
    'Datasets/Unstructured_Data/child/child.txt',
    'Datasets/Unstructured_Data/dostor_gena2y/dostor.pdf',
    'Datasets/Unstructured_Data/egra2at_gena2ya/egra2at_gena2ya.pdf',
    'Datasets/Unstructured_Data/erhab/erhab.txt',
    'Datasets/Unstructured_Data/technology/tech.txt'
]

advanced_split = [
    'Datasets/Unstructured_Data/drugs/drugs.txt',
    'Datasets/Unstructured_Data/Asle7a/asle7a.txt',
]

special_split = [
    'Datasets/Unstructured_Data/na2d/mbade2_ma7kmt_elna2d.pdf',
]

normal_patterns = {
    'law': r'قانون\s*رقم\s*\d+\s*لسنة\s*\d+',
    'book': r'(?:الكتاب|كتاب)\s*(?:الأول|الثاني|الثالث|الرابع|الخامس|السادس|السابع|الثامن|التاسع|العاشر|\d+)',
    'section': r'(?:الباب|باب)\s*(?:الأول|الثاني|الثالث|الرابع|الخامس|السادس|السابع|الثامن|التاسع|العاشر|\d+)',
    'chapter': r'(?:الفصل|فصل)\s*(?:الأول|الثاني|الثالث|الرابع|الخامس|السادس|السابع|الثامن|التاسع|العاشر|\d+)',
    'article': r'\(?\s*(?:المادة|مادة)[\s\(:\-]*([0-9٠-٩]+|الأول|الثاني|الثالث|الرابع|الخامس|السادس|السابع|الثامن|التاسع|العاشر)\s*\)?'
}

advanced_patterns = normal_patterns.copy() 

special_patterns = {
    'topic': r'^(.*?)$', 
    'summary': r'الموجز\s*:',
    'rule': r'القاعدة\s*:'
}


In [7]:
for doc in documents:
    source = doc.metadata.get('source', 'unknown')
    content = doc.page_content

    if source in normal_split:
        docs = legal_split(content, source, normal_patterns)
    elif source in advanced_split:
        docs = legal_split(content, source, advanced_patterns)
    else:
        docs = legal_split(content, source, special_patterns)
    
    all_legal_docs.extend(docs)
    print(f"{source}: {len(docs)} articles extracted")

print(f"\nTotal articles: {len(all_legal_docs)}")

Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt: 4 articles extracted
Datasets/Unstructured_Data/3okobat/3okobat_num36_2020.txt: 3 articles extracted
Datasets/Unstructured_Data/3okobat/3okobat.pdf: 512 articles extracted
Datasets/Unstructured_Data/Asle7a/asle7a.txt: 115 articles extracted
Datasets/Unstructured_Data/child/child.txt: 180 articles extracted
Datasets/Unstructured_Data/dostor_gena2y/dostor.pdf: 166 articles extracted
Datasets/Unstructured_Data/drugs/drugs.txt: 111 articles extracted
Datasets/Unstructured_Data/egra2at_gena2ya/egra2at_gena2ya.pdf: 61 articles extracted
Datasets/Unstructured_Data/erhab/erhab.txt: 67 articles extracted
Datasets/Unstructured_Data/na2d/mbade2_ma7kmt_elna2d.pdf: 6801 articles extracted
Datasets/Unstructured_Data/technology/tech.txt: 53 articles extracted

Total articles: 8073


In [9]:
for i, doc in enumerate(all_legal_docs[2000:2100]):
    example = doc.metadata
    print("\nمثال على مادة:")
    print(f"المصدر: {example.get('source', 'غير معروف')}")
    print(f"رقم المادة: {example.get('article_number', 'غير موجود')}")
    print(f"القانون: {example.get('law', 'غير موجود')}")

    if example.get('type') == 'special':
        print(f"الموضوع: {example.get('topic', '')}")
        print(f"الموجز: {example.get('summary', '')}") 
        print(f"القاعدة: {example.get('rule', '')}")

    print(f"\nالمحتوى: {doc.page_content}...") 



مثال على مادة:
المصدر: Datasets/Unstructured_Data/na2d/mbade2_ma7kmt_elna2d.pdf
رقم المادة: غير محدد
القانون: 

المحتوى: وبما مؤداه عدم تمتع االتحاد األفريقي لكرة القدم والقائمين...

مثال على مادة:
المصدر: Datasets/Unstructured_Data/na2d/mbade2_ma7kmt_elna2d.pdf
رقم المادة: غير محدد
القانون: 

المحتوى: على إدارة أنشطته بالحصانة الدبلوماسية ، األمر الذي يضحى معه قيام رجال حماية المنافسة...

مثال على مادة:
المصدر: Datasets/Unstructured_Data/na2d/mbade2_ma7kmt_elna2d.pdf
رقم المادة: غير محدد
القانون: 

المحتوى: ومنع الممارسات االحتكارية بممارسة اختصاصهم ومباشرة التحقيقات من النيابة العامة والمحاكمة...

مثال على مادة:
المصدر: Datasets/Unstructured_Data/na2d/mbade2_ma7kmt_elna2d.pdf
رقم المادة: غير محدد
القانون: 

المحتوى: ًجميعها إجراءات ال شائبة فيها ، فإن ما يثيره الطاعن في هذا الشأن ال يعدو أن يكون دفعا...

مثال على مادة:
المصدر: Datasets/Unstructured_Data/na2d/mbade2_ma7kmt_elna2d.pdf
رقم المادة: غير محدد
القانون: 

المحتوى: . قانونياً ظاهر البطالن...

مثال على مادة:
المصدر: Datasets/

In [98]:
all_legal_docs

[Document(metadata={'source': 'Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt', 'article_number': 'غير محدد', 'law': '', 'summary': '', 'rule': ''}, page_content='المادة الأولى: يستبدل بنص'),
 Document(metadata={'source': 'Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt', 'article_number': '٣٩٢', 'law': '', 'summary': '', 'rule': ''}, page_content='المادة ٣٩٢ من قانون العقوبات الصادر بالقانون رقم ٥٨ لسنة 1937 النص الآتي:'),
 Document(metadata={'source': 'Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt', 'article_number': '٣٩٢', 'law': 'قانون رقم ٥٨ لسنة 1937', 'summary': '', 'rule': ''}, page_content='مادة ٣٩٢ كل من صدر عليه حكم قضائي واجب النفاذ بدفع نفقة لزوجه أو أقاربه أو أصهاره أو أجرة حضانة أو رضاعة أو مسكن وامتنع عن الدفع مع قدرته عليه لمدة ثلاثة أشهر بعد التنبيه عليه بالدفع، يعاقب بالحبس مدة لا تزيد على سنة وبغرامة لا تتجاوز خمسة آلاف جنيه أو بإحدى هاتين العقوبتين، ولا ترفع الدعوى عليه إلا بناء على شكوى أو طلب من صاحب الشأن، وإذا رفعت بعد الحكم عليه د